# Handling Missing Data with Missing Indicators

src: [Missing Indicator | Random Sample Imputation | Handling Missing Data Part 4](https://www.youtube.com/watch?v=Ratcir3p03w)

---

## 🧾 Introduction

This notebook demonstrates a beginner-friendly technique for handling missing values using **SimpleImputer with missing indicators** in `scikit-learn`. Instead of only filling in missing values (e.g., using the mean), it also appends **binary indicator columns** that tell the model which values were originally missing.

This hybrid approach allows even simple models—like logistic regression—to detect patterns that may be **implicitly signaled by missingness** itself.

---

### 🧠 What Is This Technique?

**Missing Indicator Strategy**:

* Use `SimpleImputer(add_indicator=True)` to:

  1. **Impute missing values** (e.g., with mean, median, or a constant)
  2. **Add new columns** that flag where data was originally missing (`1 = missing`, `0 = not missing`)

---

In [47]:
# -----------------------------------------
# Import necessary libraries
# -----------------------------------------

# NumPy is a library for numerical operations (e.g., arrays, math functions)
import numpy as np

# pandas is used for handling data in tabular form (DataFrames)
import pandas as pd

# -----------------------------------------
# Import data preprocessing tools from scikit-learn
# -----------------------------------------

# train_test_split is used to split your dataset into training and test sets
from sklearn.model_selection import train_test_split

# MissingIndicator helps flag where values were missing (NaNs) in the dataset
from sklearn.impute import MissingIndicator

# SimpleImputer fills in missing values using a specified strategy (e.g., mean, median, constant)
from sklearn.impute import SimpleImputer

In [48]:
df = pd.read_csv('train.csv',usecols=['Age','Fare','Survived'])

In [49]:
df.tail()

,Survived,Age,Fare
886,0,27.0,13.00
887,1,19.0,30.00
888,0,NaN,23.45
889,1,26.0,30.00
890,0,32.0,7.75


In [50]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [51]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [52]:
X_train.tail()

,Age,Fare
534,30.0,8.6625
584,NaN,8.7125
493,71.0,49.5042
527,NaN,221.7792
168,NaN,25.9250


### 📌 Compare the accuracy before and after Imputing with Missing Indicators

---

Begin by calculating a **baseline** performance (accuracy) using:

* Simple imputation (mean strategy via `SimpleImputer`)
* No feature engineering
* No encoding for categorical features
* A basic linear model (logistic regression)

---

In [53]:
# ------------------------------------------------------
# Step 1: Create a SimpleImputer instance (default strategy is 'mean')
# ------------------------------------------------------
si = SimpleImputer()
# This will automatically replace all missing values (NaN) with the mean of each column.

# ------------------------------------------------------
# Step 2: Fit the imputer on training data and transform it
# ------------------------------------------------------
X_train_trf = si.fit_transform(X_train)
# ➤ fit_transform() does two things:
#   1. Learns the mean of each column from X_train (fit)
#   2. Replaces missing values in X_train with those means (transform)
# ➤ The output is a NumPy array where all NaNs are replaced with values

# ------------------------------------------------------
# Step 3: Apply the same transformation to the test set
# ------------------------------------------------------
X_test_trf = si.transform(X_test)
# ➤ This uses the same column means learned from training data to fill missing values in test data
# ➤ Ensures consistent treatment between train and test

**Confirm no missing values in Age column**

In [54]:
X_train_trf

array([[ 40.        ,  27.7208    ],
       [  4.        ,  16.7       ],
       [ 47.        ,   9.        ],
       ...,
       [ 71.        ,  49.5042    ],
       [ 29.78590426, 221.7792    ],
       [ 29.78590426,  25.925     ]])

In [55]:
# Convert NumPy array back into a DataFrame
X_train_trf_df = pd.DataFrame(X_train_trf, columns=X_train.columns)

# Display the first few rows to confirm
X_train_trf_df.tail()

,Age,Fare
707,30.000000,8.6625
708,29.785904,8.7125
709,71.000000,49.5042
710,29.785904,221.7792
711,29.785904,25.9250


---

> The following code trains a **logistic regression classifier** on a cleaned dataset (missing values filled in), makes predictions on the test set, and evaluates the model’s **accuracy** in predicting survival.

---


In [56]:
# ----------------------------------------
# Train a Logistic Regression Model
# ----------------------------------------

# Import the LogisticRegression model from scikit-learn
from sklearn.linear_model import LogisticRegression

# Step 1: Create a logistic regression classifier instance
clf = LogisticRegression()
# ➤ Logistic Regression is a linear model used for binary classification problems
# ➤ It estimates the probability of a data point belonging to class 1 (vs. class 0)

# Step 2: Train the model using the transformed training data (with no missing values)
clf.fit(X_train_trf, y_train)
# ➤ This fits the model by learning the relationship between the input features and the target variable (Survived)
# ➤ X_train_trf: Input features (after missing value imputation)
# ➤ y_train: Target labels (e.g., 0 = did not survive, 1 = survived)

# Step 3: Make predictions on the test set
y_pred = clf.predict(X_test_trf)
# ➤ This uses the trained model to predict outcomes for unseen test data

# ----------------------------------------
# Evaluate the Model
# ----------------------------------------

# Import accuracy_score metric to evaluate the classifier
from sklearn.metrics import accuracy_score

# Step 4: Calculate and print accuracy
accuracy_score(y_test, y_pred)
# ➤ Accuracy = (correct predictions) / (total predictions)
# ➤ Tells us how well the model performed on the test set

0.6145251396648045

👉 **Interpretation of Output (baseline accuracy):**

Out of all passengers in your test set, about 61.45% were correctly classified by the logistic regression model (whether they survived or not).

In [57]:
# ----------------------------------------
# Step 1: Create a MissingIndicator instance
# ----------------------------------------
mi = MissingIndicator()
# ➤ This object will detect which values are missing (NaN) in the dataset
# ➤ It does NOT impute/fill values — it only tracks where the missing values are

# ----------------------------------------
# Step 2: Fit the indicator to training data
# ----------------------------------------
mi.fit(X_train)
# ➤ This "learns" the structure of X_train
# ➤ It remembers which columns contain missing values and how many NaNs there are


MissingIndicator()

In [58]:
mi.features_

array([0])

---

### ✅ Interpretation of `mi.features_ = array([0])`:

> It tells us that the **first column** of your `X_train` (i.e., column index `0`) contains **missing values (`NaN`)**.

This tells us:

* The `MissingIndicator` has **identified column 0 as having missing data**
* When you call `.transform(X_train)`, it will generate a new column (or array) with:

  * `1` where the value in column 0 is missing
  * `0` where it's not missing

---

### 🧪 Example (Assume column 0 is `'Age'`):

| Age  | Fare  | **Age\_NA (Indicator)** |
| ---- | ----- | ----------------------- |
| 22.0 | 7.25  | 0                       |
| NaN  | 71.28 | 1                       |
| 35.0 | 8.05  | 0                       |

Calling:

```python
mi.transform(X_train)
```

Would output something like:

```python
array([[0],
       [1],
       [0],
       ...
])
```

---

In [59]:
X_train_missing = mi.transform(X_train)
type(X_train_missing)
# ➤ Output: <class 'numpy.ndarray'>

numpy.ndarray

In [60]:
# Completely new column representing the missing values in column 0
X_train_missing

array([[False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [

In [61]:
X_test_missing = mi.transform(X_test)

In [62]:
X_test_missing

array([[False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [ True],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [False],
       [

In [63]:
X_train.tail()

,Age,Fare
534,30.0,8.6625
584,NaN,8.7125
493,71.0,49.5042
527,NaN,221.7792
168,NaN,25.9250


In [64]:
# Assign missing values to a new column Age_NA
X_train['Age_NA'] = X_train_missing

In [65]:
X_train

,Age,Fare,Age_NA
30,40.0,27.7208,False
10,4.0,16.7000,False
873,47.0,9.0000,False
182,9.0,31.3875,False
876,20.0,9.8458,False
...,...,...,...
534,30.0,8.6625,False
584,NaN,8.7125,True
493,71.0,49.5042,False
527,NaN,221.7792,True


In [66]:
X_test

,Age,Fare
707,42.0,26.2875
37,21.0,8.0500
615,24.0,65.0000
169,28.0,56.4958
68,17.0,7.9250
...,...,...
89,24.0,8.0500
80,22.0,9.0000
846,NaN,69.5500
870,26.0,7.8958


In [67]:
X_test['Age_NA'] = X_test_missing

In [68]:
X_train

,Age,Fare,Age_NA
30,40.0,27.7208,False
10,4.0,16.7000,False
873,47.0,9.0000,False
182,9.0,31.3875,False
876,20.0,9.8458,False
...,...,...,...
534,30.0,8.6625,False
584,NaN,8.7125,True
493,71.0,49.5042,False
527,NaN,221.7792,True


### 📌 Replace all missing values in Age column with Mean

In [69]:
# ----------------------------------------
# Step 1: Create a SimpleImputer instance
# ----------------------------------------
si = SimpleImputer()
# ➤ By default, this uses strategy='mean'
# ➤ It will replace all missing values (NaN) with the mean of the corresponding column

# ----------------------------------------
# Step 2: Fit the imputer on training data and transform it
# ----------------------------------------
X_train_trf2 = si.fit_transform(X_train)
# ➤ This does two things:
#     1. Learns the column-wise means from X_train (fit)
#     2. Replaces missing values in X_train using those means (transform)
# ➤ Returns a NumPy array with no missing values

# ----------------------------------------
# Step 3: Apply the same transformation to the test data
# ----------------------------------------
X_test_trf2 = si.transform(X_test)
# ➤ Uses the column means from training data to impute missing values in test data
# ➤ Ensures consistency between train and test sets


In [70]:
# If need to view as a dataframe ...

# Replace with actual column names
# columns = X_train.columns

# X_train_trf2 = pd.DataFrame(X_train_trf2, columns=columns)
# X_test_trf2 = pd.DataFrame(X_test_trf2, columns=columns)

In [71]:
# ----------------------------------------
# Train a Logistic Regression Classifier
# ----------------------------------------

# Import the logistic regression model
from sklearn.linear_model import LogisticRegression

# Step 1: Create the model
clf = LogisticRegression()
# ➤ This is a linear model used for binary classification tasks (e.g., survived vs. not survived)

# Step 2: Train the model on the imputed training data
clf.fit(X_train_trf2, y_train)
# ➤ Learns the relationship between the features (with no missing values) and the target labels (0 or 1)

# Step 3: Make predictions on the imputed test set
y_pred = clf.predict(X_test_trf2)
# ➤ Outputs predicted class labels (0 or 1) for each example in the test set

# ----------------------------------------
# Evaluate Model Accuracy
# ----------------------------------------

# Import the accuracy metric
from sklearn.metrics import accuracy_score

# Step 4: Calculate accuracy of predictions
accuracy_score(y_test, y_pred)
# ➤ Compares predicted labels (y_pred) to actual test labels (y_test)
# ➤ Outputs a score between 0 and 1 (e.g., 0.78 = 78% correct)

0.6312849162011173

Accuracy improves slightly from 61.45% to 63.13%

**✅ Final Verdict:**

If you are able to tell the model, which rows have missing values using Missing Indicator, then accuracy improves.

### 📌 **Using `SimpleImputer` with `add_indicator=True`**



#### ➤ Automatically generate missing indicators without a separate `MissingIndicator` step

---

Instead of manually using the `MissingIndicator` class to create binary columns that flag missing values, you can use the built-in `add_indicator=True` parameter of the `SimpleImputer` class.

This approach:

* **Fills missing values** (e.g., with mean, median, constant)
* **Appends binary columns** to the output, indicating where missing values originally existed

---

In [72]:
si = SimpleImputer(add_indicator=True)

In [73]:
X_train = si.fit_transform(X_train)

In [74]:
X_test = si.transform(X_test)

In [75]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()

clf.fit(X_train_trf2,y_train)

y_pred = clf.predict(X_test_trf2)

from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.6312849162011173

---

### 📌 Interpretation:

* Your model correctly predicted the outcome (e.g., survived or not) in **\~63.13% of test cases**.
* This is **slightly better** than your earlier baseline (≈61.45%) that used `SimpleImputer` without indicators.

---

### 🧠 What Made the Difference?

By setting:

```python
SimpleImputer(add_indicator=True)
```

You improved the model’s performance by:

* **Not just filling missing values**, but also…
* **Informing the model** about where values were missing.

This helps especially when **the fact that something is missing** (like 'Age' or 'Fare') **carries meaning** — e.g., missing `Fare` might indicate a child passenger, or a complimentary ticket.

---

### 💡 Takeaway from the Instructor (based on transcript context):

> Including missing indicators often improves performance in **linear models** like Logistic Regression, because these models can’t **learn missingness patterns** the way tree-based models can.

---

🔎 **Experiment Further:**

* Visualizing the added indicator columns,
* Training a tree-based model for comparison, or
* Combining this with categorical feature encoding!

---


---

##📌 Key Takeways

### ✅ When to Use This Technique

| Use It When...                                                                                 | Avoid It When...                                                                                        |
| ---------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------- |
| You're working with **linear models**                                                          | You're using **tree-based models** like RandomForest or XGBoost — they can naturally handle missingness |
| Missingness might carry **predictive signal** (e.g., no fare might mean infant or free ticket) | The proportion of missing values is negligible or purely random                                         |
| You want the model to learn **missingness patterns** explicitly                                | You're using domain knowledge to drop or impute manually                                                |

---

### 🤖 Best Suited For These Models

* **Linear models**:

  * Logistic Regression
  * Linear Regression
* **Shallow neural nets**
* Any model where **missingness is not implicitly learned**

For **tree-based models**, this technique may not offer much improvement — trees can route missing values during splits and may even benefit from leaving NaNs untouched.

---